# Qorx 1.0.6 — DataCamp CPU verification

This notebook verifies the official Qorx language/compiler release without a GPU. It downloads the platform asset (or uses `QORX_BIN`), checks a `.qorx` program, compiles `.qorxb` bytecode, runs it, and records local timings.

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, statistics, subprocess, tarfile, time, urllib.request

VERSION = '1.0.6'
TAG = f'v{VERSION}'
REPO = 'https://github.com/bbrainfuckk/qorx'
WORK = Path.cwd() / f'qorx-{VERSION}-datacamp'
WORK.mkdir(exist_ok=True)
print({'python': platform.python_version(), 'system': platform.system(), 'machine': platform.machine()})

In [ ]:
def release_asset():
    machine = platform.machine().lower()
    arch = 'arm64' if machine in {'arm64', 'aarch64'} else 'x64'
    system = platform.system().lower()
    if system == 'windows': return f'qorx-{TAG}-windows-{arch}.zip'
    if system == 'darwin': return f'qorx-{TAG}-macos-{arch}.tar.gz'
    if system == 'linux' and arch == 'x64': return f'qorx-{TAG}-linux-x64-static.tar.gz'
    if system == 'linux': return f'qorx-{TAG}-linux-{arch}.tar.gz'
    raise RuntimeError(f'unsupported platform: {system}/{machine}')

def find_qorx(root):
    name = 'qorx.exe' if platform.system() == 'Windows' else 'qorx'
    return next((p for p in root.rglob(name) if p.is_file()), None)

override = os.environ.get('QORX_BIN')
binary = Path(override).expanduser() if override else find_qorx(WORK)
if not binary:
    asset = release_asset()
    archive = WORK / asset
    url = f'{REPO}/releases/download/{TAG}/{asset}'
    print('Downloading', url)
    urllib.request.urlretrieve(url, archive)
    checksum_file = WORK / f'{asset}.sha256'
    urllib.request.urlretrieve(f'{url}.sha256', checksum_file)
    expected = checksum_file.read_text().split()[0].lower()
    actual = hashlib.sha256(archive.read_bytes()).hexdigest()
    if actual != expected: raise RuntimeError(f'asset checksum mismatch: {actual} != {expected}')
    if asset.endswith('.zip'):
        shutil.unpack_archive(archive, WORK / 'release')
    else:
        destination = (WORK / 'release').resolve()
        with tarfile.open(archive, 'r:gz') as package:
            for member in package.getmembers():
                candidate = (destination / member.name).resolve()
                if candidate != destination and destination not in candidate.parents:
                    raise RuntimeError(f'unsafe archive member: {member.name}')
            package.extractall(destination)
    binary = find_qorx(WORK / 'release')
if not binary: raise FileNotFoundError('Set QORX_BIN or publish the matching GitHub release asset.')
binary.chmod(binary.stat().st_mode | 0o111)
print('binary:', binary, 'sha256:', hashlib.sha256(binary.read_bytes()).hexdigest())

In [ ]:
def qorx(*args, check=True):
    return subprocess.run([str(binary), *args], text=True, capture_output=True, check=check)

reported = qorx('--version').stdout.strip()
assert reported == f'qorx {VERSION}', reported
print(reported)

In [ ]:
source = WORK / 'evidence.qorx'
bytecode = WORK / 'evidence.qorxb'
source.write_text('''QORX 1
use std.evidence
use std.branch as br
let question = \"which files prove the Qorx compiler pipeline?\"
let fallback = \"local evidence does not support this answer\"
pack evidence from question budget 700
strict answer from evidence limit 2
if supported(answer) then emit answer else emit fallback
''')
print(source.read_text())

In [ ]:
checked = qorx('qorx-check', str(source))
compiled = qorx('qorx-compile', str(source), '--out', str(bytecode))
executed = qorx('qorx', str(bytecode))
assert bytecode.exists() and bytecode.stat().st_size > 0
print(checked.stdout)
print(compiled.stdout)
print(executed.stdout)

In [ ]:
samples_ms = []
for _ in range(30):
    started = time.perf_counter_ns()
    qorx('qorx-check', str(source))
    samples_ms.append((time.perf_counter_ns() - started) / 1_000_000)
ordered = sorted(samples_ms)
result = {
    'schema': 'qorx.datacamp.cpu.v1',
    'qorx_version': reported,
    'binary_sha256': hashlib.sha256(binary.read_bytes()).hexdigest(),
    'binary_bytes': binary.stat().st_size,
    'bytecode_bytes': bytecode.stat().st_size,
    'iterations': len(samples_ms),
    'check_median_ms': statistics.median(samples_ms),
    'check_p95_ms': ordered[int(0.95 * (len(ordered) - 1))],
    'platform': {'system': platform.system(), 'machine': platform.machine(), 'python': platform.python_version()},
}
result_path = WORK / f'QORX-DATACAMP-CPU-{VERSION}.json'
result_path.write_text(json.dumps(result, indent=2) + '\n')
print(json.dumps(result, indent=2))
print('saved:', result_path)